# Multi-Agent AI Business Intelligence System for E-Commerce

## Project Overview

This project implements a multi-agent AI system for analyzing e-commerce
business data and generating evidence-grounded business recommendations.

The system combines:

- Data analysis using Python and Pandas
- Machine Learning for customer segmentation and anomaly detection
- LLM-based agents for orchestration, interpretation, and recommendations
- RAG for retrieving business policies
- Automated report generation

## Main Workflow

User Query
    ↓
Orchestrator Agent
    ↓
Data Analysis Tools
    ├── Sales Tool
    ├── Customer Tool
    └── Product/Anomaly Tool
    ↓
Insight Agent
    ↓
Business Policy RAG
    ↓
Recommendation Agent
    ↓
Report Generator
    ↓
Business Intelligence Report

## Important Design Principle

Numerical analysis is performed by deterministic Python tools.

LLMs are used for:
- Task routing
- Insight generation
- Policy-aware interpretation
- Recommendation generation

This hybrid architecture improves controllability and reduces unnecessary
LLM calls.

# Final System Architecture

The final system follows a hybrid multi-agent architecture.

                    User
                      │
                      ▼
             ┌─────────────────┐
             │  Orchestrator   │
             │     Agent       │
             └────────┬────────┘
                      │
          ┌───────────┼───────────┐
          ▼           ▼           ▼
     Sales Tool  Customer Tool  Product Tool
          │           │           │
          └───────────┼───────────┘
                      ▼
             ┌─────────────────┐
             │   Insight       │
             │     Agent       │
             └────────┬────────┘
                      │
                      ▼
             ┌─────────────────┐
             │   RAG System    │
             │ Business Policy │
             └────────┬────────┘
                      │
                      ▼
             ┌─────────────────┐
             │ Recommendation  │
             │     Agent       │
             └────────┬────────┘
                      │
                      ▼
                   Report

## Design Principle

Not every component is an LLM agent.

Deterministic analytical tasks are handled by Python tools,
while LLM agents are responsible for orchestration,
interpretation, and recommendations.

This reduces unnecessary LLM calls and improves
controllability, reproducibility, and efficiency.

## Installation & Imports

In [1]:
!pip install -qU langchain langchain-community langchain-core langchain-groq \
    langchain-huggingface sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires reque

In [2]:
import os, re, time, json
import pandas as pd
import numpy as np
from getpass import getpass

from groq import RateLimitError
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_2715/3250521766.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Loading Data & Read

In [3]:
import os
from google.colab import files

def upload_agent_data_and_policy():
    os.makedirs("agent_data", exist_ok=True)

    print("📂 Upload agent_data files (business_data.csv, customer_segments.csv, daily_sales.csv, sales_anomalies.csv, kpis.json):")
    uploaded_data = files.upload()

    for name in uploaded_data.keys():
        dest = os.path.join("agent_data", name)
        os.rename(name, dest)
        print(f"✅ {name} moved to agent_data/")

    print("\n📄 Upload Business_Policies.pdf:")
    uploaded_pdf = files.upload()
    for name in uploaded_pdf.keys():
        if name != "Business_Policies.pdf":
            os.rename(name, "Business_Policies.pdf")
        print(f"✅ {name} saved as Business_Policies.pdf")

    print("\n--- File check ---")
    required = ["business_data.csv", "customer_segments.csv",
                "daily_sales.csv", "sales_anomalies.csv", "kpis.json"]
    for f in required:
        path = os.path.join("agent_data", f)
        print("✅" if os.path.exists(path) else "❌", f)
    print("✅" if os.path.exists("Business_Policies.pdf") else "❌", "Business_Policies.pdf")

upload_agent_data_and_policy()

📂 Upload agent_data files (business_data.csv, customer_segments.csv, daily_sales.csv, sales_anomalies.csv, kpis.json):


Saving business_data.csv to business_data.csv
Saving customer_segments.csv to customer_segments.csv
Saving daily_sales.csv to daily_sales.csv
Saving kpis.json to kpis.json
Saving sales_anomalies.csv to sales_anomalies.csv
✅ business_data.csv moved to agent_data/
✅ customer_segments.csv moved to agent_data/
✅ daily_sales.csv moved to agent_data/
✅ kpis.json moved to agent_data/
✅ sales_anomalies.csv moved to agent_data/

📄 Upload Business_Policies.pdf:


Saving Business_Policies.pdf to Business_Policies.pdf
✅ Business_Policies.pdf saved as Business_Policies.pdf

--- File check ---
✅ business_data.csv
✅ customer_segments.csv
✅ daily_sales.csv
✅ sales_anomalies.csv
✅ kpis.json
✅ Business_Policies.pdf


In [4]:
required = ["business_data.csv", "customer_segments.csv",
            "daily_sales.csv", "sales_anomalies.csv", "kpis.json"]
for f in required:
    print("✅" if os.path.exists(f"agent_data/{f}") else "❌", f)

business_data     = pd.read_csv("agent_data/business_data.csv")
customer_segments = pd.read_csv("agent_data/customer_segments.csv")
daily_sales       = pd.read_csv("agent_data/daily_sales.csv")
sales_anomalies   = pd.read_csv("agent_data/sales_anomalies.csv")
with open("agent_data/kpis.json") as f:
    kpis = json.load(f)

print(business_data.shape, customer_segments.shape, daily_sales.shape, sales_anomalies.shape)
print(kpis)

✅ business_data.csv
✅ customer_segments.csv
✅ daily_sales.csv
✅ sales_anomalies.csv
✅ kpis.json
(113425, 30) (96095, 4) (633, 3) (19, 3)
{'total_revenue': 16008872.12, 'total_orders': 99441, 'total_customers': 96096, 'average_order_value': 160.98864774087147}


## RAG (load + split + FAISS)

In [5]:
documents = PyPDFLoader("Business_Policies.pdf").load()

#ٌ Recursive for Meaning
chunks = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=100
).split_documents(documents)
print("Pages:", len(documents), "| Chunks:", len(chunks))

# Create Embeddings and vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Vector store ready")

Pages: 5 | Chunks: 10


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store ready


## Test Retrival

In [6]:
for i, doc in enumerate(retriever.invoke("What is the policy for discounts above 20%?"), 1):
    print(f"--- Result {i} ---\n{doc.page_content}\n")

--- Result 1 ---
2.3 Repeated Delivery Delays
Repeated delivery delays associated with the same seller should be investigated.
2.4 Delivery Issues
Customers reporting delivery problems should receive appropriate support based on the order status
and delivery information.
3. Discount Policy
3.1 Standard Discounts
Discounts up to 10% may be applied through standard promotional campaigns.
3.2 Large Discounts
Discounts above 20% require manager approval.
3.3 Unusual Discounts
Unusually large discounts should be reviewed before being applied.
3.4 Promotional Campaigns
Promotional campaigns should be monitored to ensure that discounts do not negatively affect
business performance.
4. Customer Service Policy
4.1 Customer Complaints

--- Result 2 ---
7. Recommendations should consider both analytical results and applicable business policies.
8. Example Business Scenarios
Scenario 1 — Delayed Order
Situation: An order has experienced a significant delivery delay.
Relevant Policies: Shipping Pol

## Calling LLM

Groq API_Key

In [7]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


LLMs (llm for General use & llm_fast for Orchestrator Agent use)

In [8]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=1500,
    reasoning_effort="medium",  # بدل الافتراضي (اللي كان بيستهلك رصيد كبير)
)

llm_fast = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=500,
    reasoning_effort="low",
)

def safe_invoke(chain, inputs, retries=8, max_wait=600, verbose=False):
    for attempt in range(retries):
        try:
            return chain.invoke(inputs)
        except RateLimitError as e:
            msg = str(e)
            m = re.search(r"try again in (?:(\d+)m)?([\d.]+)s", msg)
            if m:
                minutes = float(m.group(1)) if m.group(1) else 0
                seconds = float(m.group(2))
                wait = min(minutes * 60 + seconds + 1, max_wait)
            else:
                wait = min(15 * (attempt + 1), max_wait)

            is_daily = "tokens per day" in msg or "TPD" in msg
            if verbose or wait > 60:
                label = "Daily limit reached" if is_daily else "Rate limit"
                print(f"⏳ {label}, waiting {wait/60:.1f} min...")

            time.sleep(wait)
    raise RuntimeError("Rate limit: retries exhausted")

print("LLMs ready")

LLMs ready


## Tools

Policy tool

In [9]:
@tool
def business_policy_search(query: str) -> str:
    """Search the business policy knowledge base for relevant rules and policies."""
    results = retriever.invoke(query)
    if not results:
        return "No relevant business policy was found."
    return "\n\n".join(d.page_content for d in results)

Sales tool

In [10]:
@tool
def sales_analysis() -> str:
    """Analyze overall sales performance and return important business KPIs."""
    cat = (business_data.groupby("product_category_name_english")["price"]
           .sum().sort_values(ascending=False))
    items_total = business_data["price"].sum()
    top5 = cat.head(5)
    repeat_n = int((customer_segments["orders"] > 1).sum())

    return json.dumps({
        "total_revenue": round(kpis["total_revenue"], 2),
        "total_orders": kpis["total_orders"],
        "total_customers": kpis["total_customers"],
        "average_order_value": round(kpis["average_order_value"], 2),
        "repeat_customers": repeat_n,
        "repeat_customers_pct": round(repeat_n / len(customer_segments) * 100, 2),
        "top_categories_item_sales": top5.round(2).to_dict(),
        "top_categories_share_of_item_sales_pct": (top5 / items_total * 100).round(2).to_dict(),
        "top5_combined_item_sales": round(top5.sum(), 2),
        "top5_combined_share_of_item_sales_pct": round(top5.sum() / items_total * 100, 2),
        "note": "Shares are % of product item sales (sum of price, excl. freight). "
                "They are NOT % of total_revenue, which comes from payments and includes freight.",
    })

Customer Tool

In [11]:
@tool
def customer_analysis() -> str:
    """Analyze customer segments and spending behavior."""
    s = (customer_segments.groupby("segment")
         .agg(customers=("customer_unique_id", "count"),
              total_spending=("spending", "sum"),
              average_spending=("spending", "mean"),
              average_orders=("orders", "mean"))
         .reset_index())
    s["pct_customers"] = (s["customers"] / s["customers"].sum() * 100).round(1)
    s["pct_revenue"] = (s["total_spending"] / s["total_spending"].sum() * 100).round(1)
    s["value_index"] = (s["pct_revenue"] / s["pct_customers"]).round(2)

    high_value = s[s["value_index"] > 1]

    result = {
        "segments": s.round(2).to_dict(orient="records"),
        "high_value_segments_combined_pct_customers": round(high_value["pct_customers"].sum(), 2),
        "high_value_segments_combined_pct_revenue": round(high_value["pct_revenue"].sum(), 2),
        "note": "high_value_segments are those with value_index > 1; combined figures are precomputed, do not recompute.",
    }
    return json.dumps(result)

Product Tool

In [12]:
@tool
def product_analysis() -> str:
    """Analyze product category performance and unusual sales days."""
    cat = (business_data.groupby("product_category_name_english")["price"]
           .sum().sort_values(ascending=False))
    items_total = business_data["price"].sum()
    top = cat.head(5)

    an = sales_anomalies.copy()
    median_day = daily_sales["payment_value"].median()
    an["direction"] = np.where(an["payment_value"] > median_day, "high", "low")
    an["month"] = pd.to_datetime(an["date"]).dt.strftime("%Y-%m")

    high8 = an[an["direction"] == "high"].sort_values("payment_value", ascending=False).head(8)
    low5 = an[an["direction"] == "low"].sort_values("payment_value").head(5)

    return json.dumps({
        "top_categories_item_sales": top.round(2).to_dict(),
        "top5_combined_share_of_item_sales_pct": round(top.sum() / items_total * 100, 2),
        "data_range": [str(daily_sales["date"].min()), str(daily_sales["date"].max())],
        "median_daily_payment": round(median_day, 2),
        "number_of_sales_anomalies": len(an),
        "high_anomaly_days": int((an["direction"] == "high").sum()),
        "low_anomaly_days": int((an["direction"] == "low").sum()),
        "anomalies_per_month": an["month"].value_counts().sort_index().to_dict(),
        "top_high_anomalous_days": high8[["date", "payment_value"]].to_dict(orient="records"),
        "lowest_anomalous_days": low5[["date", "payment_value"]].to_dict(orient="records"),
        "note": "Anomaly values are TOTAL DAILY payments (positive numbers), not single transactions. "
                "Direction is relative to the median daily payment. "
                "Shares are % of product item sales, not total revenue.",
    }, default=str)

## Create the Orchestrator Agent

In [13]:
orchestrator_prompt = ChatPromptTemplate.from_template("""
You are the Orchestrator Agent of an e-commerce Business Intelligence system.
Decide which tools are required for the user's request.

Tools:
SALES    - revenue, orders, customers, average order value, top categories
CUSTOMER - customer segments, spending behavior
PRODUCT  - product category performance, sales anomalies
POLICY   - business rules: returns, discounts, shipping, escalation

Return ONLY the tool names separated by commas.

Examples:
"How are our sales performing?" -> SALES
"Which customer segments are most valuable?" -> CUSTOMER
"Why should we investigate unusual sales?" -> PRODUCT
"Can we give a 25% discount?" -> POLICY
"Analyze sales and recommend actions according to our policies." -> SALES, PRODUCT, POLICY

User Request:
{query}
""")

orchestrator_chain = orchestrator_prompt | llm_fast | StrOutputParser()

print(safe_invoke(orchestrator_chain, {"query": "Analyze our sales and identify unusual sales patterns."}))

SALES, PRODUCT


## Run Tools

In [14]:
def run_tools(decision, query):
    d = decision.upper()
    results = {}
    if "SALES" in d:    results["sales"]    = sales_analysis.invoke({})
    if "CUSTOMER" in d: results["customer"] = customer_analysis.invoke({})
    if "PRODUCT" in d:  results["product"]  = product_analysis.invoke({})
    if "POLICY" in d:   results["policy"]   = business_policy_search.invoke({"query": query})
    return results

## Insight Agent

In [15]:
insight_prompt = ChatPromptTemplate.from_template("""
You are the Insight Agent in an e-commerce Business Intelligence system.

Rules:
- Identify important business patterns and explain unusual results.
- Do not invent numbers. Use ONLY numbers that appear in the results.
- Do not add, subtract or derive new numbers, percentages, or counts.
- If the results contain high_value_segments_combined_pct_customers / _pct_revenue, use those values directly; do not sum individual segment percentages yourself.
- Do not classify an anomaly as fraud or an error without evidence.
- Clearly separate facts from interpretations.
- Explanations that use outside knowledge (holidays, campaigns) must be labeled "Hypothesis".
- Segments are unnamed numbers; describe them by their numbers only.
- Do not claim causes or links between metrics unless the results show them (AOV is revenue/orders, so it never "confirms" anything).
- Do not mention margins or profitability; there is no margin data.
- Category shares are % of product item sales. Never call them "% of total revenue".
- Anomaly values are total daily payments (positive), not single transactions.
- Low anomalies near the start of data_range may reflect low early volume rather than weak sales; label that as Hypothesis.
- Say "bulk of revenue" only if a share is above 50%.
- value_index = pct_revenue / pct_customers; above 1 means the segment earns more than its size.
- Anomalies can be "high" or "low" days; treat them separately.
- Category totals and anomaly days are independent results; do not link them.
- Quote dates exactly as given, and count only items actually listed.
- Keep the answer under 250 words.

Analytical Results:
{results}

Provide a concise business insight report.
""")

insight_chain = insight_prompt | llm | StrOutputParser()

## Recommendation Agent

In [16]:
recommendation_prompt = ChatPromptTemplate.from_template("""
You are the Recommendation Agent.
Generate practical recommendations based on the results, insights, and policies.

Rules:
- Do not invent information; recommendations must be supported by the data.
- Do not create findings about plans or risks that are not in the data.
- If the results contain high_value_segments_combined_pct_customers / _pct_revenue, use those values directly; do not sum individual segment percentages yourself.
- Use business policies when applicable, and cite the policy number.
- Policies are rules, never describe a policy scenario as something that happened.
- "Required Approval": write any approval, escalation or review that a cited policy requires (with its number). Otherwise write "None".
- Do not treat anomalies as confirmed problems.
- Do not invent thresholds, alert levels, or numbers. Use ONLY numbers that appear in the results; never derive new ones.
- Do not claim margins, pricing consistency, or causes.
- If a finding needs data we don't have (inventory, margins), say "Needs data" instead of assuming.
- Give at most 3 findings, under 250 words total.

Analytical Results:
{results}

Business Insights:
{insights}

Relevant Policies:
{policies}

For each finding write: Key Finding, Recommended Action, Business Reason, Required Approval.
""")

recommendation_chain = recommendation_prompt | llm | StrOutputParser()

## Policy Answer chain

In [17]:
policy_answer_prompt = ChatPromptTemplate.from_template("""
Answer the user's question using ONLY the policy text below.
Cite the policy number (e.g. 3.2). If the policy text does not cover it, say so.
Lead with the direct answer. If approval or review is required, say that first instead of "Yes".
Policies are rules, not observations: do not describe events or problems.
Keep it under 120 words.

Policy text:
{policies}

Question: {query}
""")

policy_answer_chain = policy_answer_prompt | llm | StrOutputParser()

## Pipeline

policy retrieval

In [18]:
POLICY_TOPICS = [
    "discount limits and manager approval",
    "escalation rules for repeated anomalies",
    "high-value customer priority",
    "inventory replenishment for fast-moving products",
]

def policy_context(query=None, k=2):
    seen, out = set(), []
    for topic in POLICY_TOPICS:
        for d in vectorstore.similarity_search(topic, k=k):
            if d.page_content not in seen:
                seen.add(d.page_content)
                out.append(d.page_content)
    return "\n\n".join(out)

In [19]:
def run_business_intelligence(query):
    print("USER REQUEST\n", query.strip(), "\n" + "=" * 60)

    decision = safe_invoke(orchestrator_chain, {"query": query})
    print("\nORCHESTRATOR DECISION\n", decision)

    wants_policy = "POLICY" in decision.upper()
    results = run_tools(decision, query)
    print("\nTOOLS EXECUTED:", ", ".join(f"✅ {k}" for k in results))

    policies = results.pop("policy", None) or business_policy_search.invoke({"query": query})

    # سؤال policy بس → إجابة مباشرة
    if not results:
        answer = safe_invoke(policy_answer_chain, {"policies": policies, "query": query})
        print("\nPOLICY ANSWER\n", answer)
        return {"decision": decision, "results": {"policy": policies},
                "insights": "", "recommendations": answer}

    results_text = json.dumps(results, default=str)

    insights = safe_invoke(insight_chain, {"results": results_text})
    print("\nINSIGHT AGENT\n", insights)

    if wants_policy:
        # المستخدم طلب policies صراحة → سياق أوسع بمواضيع متعددة
        policies = policy_context(query)
    else:
        extra = business_policy_search.invoke({"query": insights[:400]})
        if extra != policies:
            policies = f"{policies}\n\n{extra}"

    recommendations = safe_invoke(recommendation_chain, {
        "results": results_text, "insights": insights, "policies": policies})
    print("\nRECOMMENDATION AGENT\n", recommendations)

    return {"decision": decision, "results": results,
            "insights": insights, "recommendations": recommendations}

## Run the Full System

In [20]:
result = run_business_intelligence("""
Analyze our e-commerce business performance, identify unusual sales patterns,
and recommend actions according to the business policies.
""")

USER REQUEST
 Analyze our e-commerce business performance, identify unusual sales patterns,
and recommend actions according to the business policies. 

ORCHESTRATOR DECISION
 SALES, PRODUCT, POLICY

TOOLS EXECUTED: ✅ sales, ✅ product, ✅ policy

INSIGHT AGENT
 **Facts**  
- Total revenue: **$16,008,872.12**; total orders: **99,441**; total customers: **96,096**.  
- Average order value (AOV): **$160.99**.  
- Repeat customers: **2,997** (3.12 % of customers).  
- Item‑sales totals for the five largest categories:  
  - health_beauty $1,258,681.34 (9.26 % of item sales)  
  - watches_gifts $1,205,005.68 (8.87 %)  
  - bed_bath_table $1,036,988.68 (7.63 %)  
  - sports_leisure $988,048.97 (7.27 %)  
  - computers_accessories $911,954.32 (6.71 %)  
- These five categories together account for **$5,400,678.99**, which is **39.74 %** of total item sales.  
- Data span: **2016‑09‑04 to 2018‑10‑17**. Median daily payment: **$23,559.52**.  
- Payment‑level anomalies: **19** days (17 high, 2 low

## Create Automated Business Report (Automation part)

In [21]:
results_json = json.dumps(result["results"], indent=2, default=str)

report = f"""# E-Commerce Business Intelligence Report

## Orchestrator Decision
{result["decision"]}

## Analytical Results
{results_json}

## Business Insights
{result["insights"]}

## Recommendations
{result["recommendations"]}

---
Generated automatically by the Multi-Agent AI Business Intelligence System.
"""

with open("business_intelligence_report.md", "w", encoding="utf-8") as f:
    f.write(report)
print("Report saved")

Report saved


## Download the Report

In [22]:
# from google.colab import files
# files.download("business_intelligence_report.md")

## Testing & Evaluation

Run Test Cases

In [23]:
test_cases = [
    "Analyze our overall sales performance.",
    "Which customer segments are most valuable?",
    "Identify unusual sales patterns.",
    "Can we offer a 25% discount?",
    "Analyze our sales performance and recommend actions according to business policies.",
]

for i, q in enumerate(test_cases, 1):
    print("\n" + "=" * 70 + f"\nTEST CASE {i}\n" + "=" * 70)
    try:
        run_business_intelligence(q)
        print("\nTEST COMPLETED")
    except Exception as e:
        print("❌ FAILED:", e)
    time.sleep(30)


TEST CASE 1
USER REQUEST
 Analyze our overall sales performance. 

ORCHESTRATOR DECISION
 SALES

TOOLS EXECUTED: ✅ sales

INSIGHT AGENT
 **Facts**  
- Total revenue: **$16,008,872.12**  
- Total orders: **99,441** → average order value **$160.99** (revenue ÷ orders)  
- Total customers: **96,096**; repeat customers: **2,997** (**3.12 %** of customers)  
- Item‑sales (price only, no freight) for the five highest‑selling categories:  
  - Health & Beauty: **$1,258,681.34** (**9.26 %** of item sales)  
  - Watches & Gifts: **$1,205,005.68** (**8.87 %**)  
  - Bed, Bath & Table: **$1,036,988.68** (**7.63 %**)  
  - Sports & Leisure: **$988,048.97** (**7.27 %**)  
  - Computers & Accessories: **$911,954.32** (**6.71 %**)  
- Combined item sales of these five categories: **$5,400,678.99**, representing **39.74 %** of total item sales.  

**Interpretations**  
- The top five categories together account for less than half of item sales, indicating a **diverse product mix** where the remaining

Ask Function

In [24]:
def ask(query):
    print("QUERY:", query if query else "(empty)")
    print("=" * 60)
    try:
        result = run_business_intelligence(query)
        print("\nRECOMMENDATIONS:\n", result.get("recommendations"))
    except Exception as e:
        print("❌ FAILED:", e)

In [25]:
ask("What's our total revenue this period?")

QUERY: What's our total revenue this period?
USER REQUEST
 What's our total revenue this period? 

ORCHESTRATOR DECISION
 SALES

TOOLS EXECUTED: ✅ sales

INSIGHT AGENT
 **Facts**  
- Total revenue: **$16,008,872.12**  
- Total orders: **99,441** → average order value **$160.99** (revenue ÷ orders)  
- Total customers: **96,096**; repeat customers: **2,997** (**3.12 %** of customers)  
- Item‑sales by top categories:  
  - Health & Beauty: **$1,258,681.34** (**9.26 %** of item sales)  
  - Watches & Gifts: **$1,205,005.68** (**8.87 %**)  
  - Bed, Bath & Table: **$1,036,988.68** (**7.63 %**)  
  - Sports & Leisure: **$988,048.97** (**7.27 %**)  
  - Computers & Accessories: **$911,954.32** (**6.71 %**)  
- Combined sales of these five categories: **$5,400,678.99**, representing **39.74 %** of total item sales.  

**Interpretations**  
- The top five categories together account for less than half of item sales, indicating a **diverse product mix** where the remaining 60 % of item sales a

In [26]:
ask("How many orders and customers do we have?")

QUERY: How many orders and customers do we have?
USER REQUEST
 How many orders and customers do we have? 

ORCHESTRATOR DECISION
 SALES

TOOLS EXECUTED: ✅ sales

INSIGHT AGENT
 **Facts**  
- Total revenue: **$16,008,872.12**  
- Total orders: **99,441** → average order value **$160.99** (revenue ÷ orders)  
- Total customers: **96,096**; repeat customers: **2,997** (**3.12 %** of customers)  
- Item‑sales (price only, no freight) for the five highest‑selling categories:  
  - Health & Beauty: **$1,258,681.34** (**9.26 %** of item sales)  
  - Watches & Gifts: **$1,205,005.68** (**8.87 %**)  
  - Bed, Bath & Table: **$1,036,988.68** (**7.63 %**)  
  - Sports & Leisure: **$988,048.97** (**7.27 %**)  
  - Computers & Accessories: **$911,954.32** (**6.71 %**)  
- Combined item sales of these five categories: **$5,400,678.99**, representing **39.74 %** of total item sales.  

**Interpretations**  
- The top five categories together account for less than half of item sales, indicating a **di

In [ ]:
# ask("What is the average order value?")

QUERY: What is the average order value?
USER REQUEST
 What is the average order value? 

ORCHESTRATOR DECISION
 SALES

TOOLS EXECUTED: ✅ sales

INSIGHT AGENT
 **Business Insight Report**

**Key Facts**
*   **Scale:** Total revenue is $16,008,872.12 from 99,441 orders.
*   **Customer Base:** 96,096 total customers, with 2,997 repeat customers (3.12%).
*   **Order Value:** The average order value is $160.99.
*   **Category Performance:** The top five categories by product item sales are:
    1.  Health & Beauty: $1,258,681.34 (9.26% share)
    2.  Watches & Gifts: $1,205,005.68 (8.87% share)
    3.  Bed, Bath & Table: $1,036,988.68 (7.63% share)
    4.  Sports & Leisure: $988,048.97 (7.27% share)
    5.  Computers & Accessories: $911,954.32 (6.71% share)
*   **Concentration:** These top five categories combined account for $5,400,678.99 in item sales, representing 39.74% of product item sales.

**Interpretations**
*   **Low Retention:** With only 3.12% of customers being repeat buyers, t

In [ ]:
# ask("How many customer segments do we have?")

QUERY: How many customer segments do we have?
USER REQUEST
 How many customer segments do we have? 

ORCHESTRATOR DECISION
 CUSTOMER

TOOLS EXECUTED: ✅ customer

INSIGHT AGENT
 **Customer Segment Insights**

**Facts**
The customer base is divided into four segments. Segment 0 is the largest, comprising 88.0% of customers and generating 59.5% of revenue. Its average spending is 112.58, with a value index of 0.68.

Segments 1, 2, and 3 are classified as high-value (value index > 1). Together, these segments represent 12.1% of customers but contribute 40.6% of total revenue.

*   **Segment 1:** 8.1% of customers, 25.7% of revenue. Average spending is 531.15. Value index is 3.17.
*   **Segment 2:** 3.1% of customers, 5.4% of revenue. Average spending is 290.34. This segment has the highest average order frequency at 2.12 orders per customer. Value index is 1.74.
*   **Segment 3:** 0.9% of customers, 9.5% of revenue. This is the smallest segment by customer count but has the highest average

In [ ]:
# ask("Which segment has the highest average spending?")

QUERY: Which segment has the highest average spending?
USER REQUEST
 Which segment has the highest average spending? 

ORCHESTRATOR DECISION
 CUSTOMER

TOOLS EXECUTED: ✅ customer

INSIGHT AGENT
 **Customer Segment Insights**

**Facts**
The customer base is divided into four segments. Segment 0 is the largest, comprising 88.0% of customers and generating 59.5% of revenue. Its average spending is 112.58, with a value index of 0.68.

Segments 1, 2, and 3 are classified as high-value (value index > 1). Together, these segments represent 12.1% of customers but contribute 40.6% of total revenue.

*   **Segment 1:** 8.1% of customers, 25.7% of revenue. Average spending is 531.15. Value index is 3.17.
*   **Segment 2:** 3.1% of customers, 5.4% of revenue. Average spending is 290.34. This segment has the highest average order frequency at 2.12 orders per customer. Value index is 1.74.
*   **Segment 3:** 0.9% of customers, 9.5% of revenue. This is the smallest segment by customer count but has t

In [ ]:
# ask("Tell me about our customer base.")

QUERY: Tell me about our customer base.
USER REQUEST
 Tell me about our customer base. 

ORCHESTRATOR DECISION
 CUSTOMER

TOOLS EXECUTED: ✅ customer

INSIGHT AGENT
 **Customer Segment Insights**

**Facts:**
*   **Segment 0** represents the majority of the customer base, comprising 88.0% of customers and 59.5% of revenue. Its value index is 0.68, indicating it generates less revenue relative to its size.
*   **Segment 1** accounts for 8.1% of customers but contributes 25.7% of revenue, with a value index of 3.17.
*   **Segment 2** comprises 3.1% of customers and 5.4% of revenue, with a value index of 1.74.
*   **Segment 3** is the smallest group at 0.9% of customers but drives 9.5% of revenue, showing the highest value index at 10.56.
*   High-value segments (those with a value index > 1) collectively represent 12.1% of customers and 40.6% of revenue.

**Interpretations:**
*   There is a significant disparity in customer value. While Segment 0 dominates volume, it is less efficient in r

In [ ]:
# ask("Were there any unusual spikes in sales?")

QUERY: Were there any unusual spikes in sales?
USER REQUEST
 Were there any unusual spikes in sales? 

ORCHESTRATOR DECISION
 SALES, PRODUCT

TOOLS EXECUTED: ✅ sales, ✅ product

INSIGHT AGENT
 **Business Insight Report**

**Sales Overview**
The business generated **$16,008,872.12** in total revenue from **99,441** orders, resulting in an average order value of **$160.99**. Customer retention is low, with only **3.12%** of the **96,096** total customers identified as repeat buyers (**2,997** customers).

**Category Performance**
No single category dominates the market. The top five categories account for **39.74%** of product item sales, indicating a diversified portfolio.
*   **Health & Beauty** leads with **$1,258,681.34** in item sales (**9.26%** share).
*   **Watches & Gifts** follows closely at **$1,205,005.68** (**8.87%** share).
*   **Bed, Bath & Table** ranks third with **$1,036,988.68** (**7.63%** share).
*   **Sports & Leisure** and **Computers & Accessories** contribute **$98

In [ ]:
# ask("What are our best-selling product categories?")

QUERY: What are our best-selling product categories?
USER REQUEST
 What are our best-selling product categories? 

ORCHESTRATOR DECISION
 SALES, PRODUCT

TOOLS EXECUTED: ✅ sales, ✅ product

INSIGHT AGENT
 **Business Insight Report**

**Sales Overview**
The business generated **$16,008,872.12** in total revenue from **99,441** orders, resulting in an average order value of **$160.99**. Customer retention is low, with only **3.12%** of the **96,096** total customers identified as repeat buyers (**2,997** customers).

**Category Performance**
No single category dominates the market. The top five categories account for **39.74%** of product item sales, indicating a diversified portfolio.
*   **Health & Beauty** leads with **$1,258,681.34** in item sales (**9.26%** share).
*   **Watches & Gifts** follows closely at **$1,205,005.68** (**8.87%** share).
*   **Bed, Bath & Table** ranks third with **$1,036,988.68** (**7.63%** share).
*   **Sports & Leisure** and **Computers & Accessories** cont

In [ ]:
# ask("Did we have any days with unusually low sales?")

QUERY: Did we have any days with unusually low sales?
USER REQUEST
 Did we have any days with unusually low sales? 

ORCHESTRATOR DECISION
 SALES, PRODUCT

TOOLS EXECUTED: ✅ sales, ✅ product

INSIGHT AGENT
 **Business Insight Report**

**Sales Overview**
The business generated **$16,008,872.12** in total revenue from **99,441** orders, resulting in an average order value of **$160.99**. Customer retention is low, with only **3.12%** of the **96,096** total customers identified as repeat buyers (**2,997** customers).

**Category Performance**
Product item sales are distributed across multiple categories, with no single category dominating. The top five categories account for **39.74%** of product item sales.
*   **health_beauty**: $1,258,681.34 (9.26% share)
*   **watches_gifts**: $1,205,005.68 (8.87% share)
*   **bed_bath_table**: $1,036,988.68 (7.63% share)
*   **sports_leisure**: $988,048.97 (7.27% share)
*   **computers_accessories**: $911,954.32 (6.71% share)

*Note: These shares r

In [ ]:
# ask("What's our policy on delivery delays?")

QUERY: What's our policy on delivery delays?
USER REQUEST
 What's our policy on delivery delays? 

ORCHESTRATOR DECISION
 POLICY

TOOLS EXECUTED: ✅ policy

POLICY ANSWER
 Orders with significantly delayed delivery should be flagged for review (Policy 2.2). Additionally, repeated delivery delays associated with the same seller should be investigated (Policy 2.3).

RECOMMENDATIONS:
 Orders with significantly delayed delivery should be flagged for review (Policy 2.2). Additionally, repeated delivery delays associated with the same seller should be investigated (Policy 2.3).


In [ ]:
# ask("Can we offer a 5% discount without approval?")

QUERY: Can we offer a 5% discount without approval?
USER REQUEST
 Can we offer a 5% discount without approval? 

ORCHESTRATOR DECISION
 POLICY

TOOLS EXECUTED: ✅ policy

POLICY ANSWER
 Yes, a 5% discount can be offered without approval. Policy 3.1 states that discounts up to 10% may be applied through standard promotional campaigns, which does not require manager approval. Manager approval is only required for discounts above 20% per Policy 3.2.

RECOMMENDATIONS:
 Yes, a 5% discount can be offered without approval. Policy 3.1 states that discounts up to 10% may be applied through standard promotional campaigns, which does not require manager approval. Manager approval is only required for discounts above 20% per Policy 3.2.


In [ ]:
# ask("What happens if a high-value customer complains?")

QUERY: What happens if a high-value customer complains?
USER REQUEST
 What happens if a high-value customer complains? 

ORCHESTRATOR DECISION
 CUSTOMER, POLICY

TOOLS EXECUTED: ✅ customer, ✅ policy

INSIGHT AGENT
 **Customer Segment Insights**

**Facts**
The customer base is divided into four segments. Segment 0 is the largest, comprising 88.0% of customers and generating 59.5% of revenue. Its average spending is 112.58, with a value index of 0.68.

Segments 1, 2, and 3 are classified as high-value (value index > 1). Together, these segments represent 12.1% of customers but contribute 40.6% of total revenue.

*   **Segment 1:** 8.1% of customers, 25.7% of revenue. Average spending is 531.15. Value index is 3.17.
*   **Segment 2:** 3.1% of customers, 5.4% of revenue. Average spending is 290.34. This segment has the highest average order frequency at 2.12 orders per customer. Value index is 1.74.
*   **Segment 3:** 0.9% of customers, 9.5% of revenue. This is the smallest segment by cust

In [ ]:
# ask("What's the policy for fast-moving products with low stock?")

QUERY: What's the policy for fast-moving products with low stock?
USER REQUEST
 What's the policy for fast-moving products with low stock? 

ORCHESTRATOR DECISION
 PRODUCT, POLICY

TOOLS EXECUTED: ✅ product, ✅ policy

INSIGHT AGENT
 **Business Insight Report**

**Category Performance**
The top five categories account for 39.74% of product item sales. Health & Beauty leads with $1,258,681.34, followed by Watches & Gifts ($1,205,005.68), Bed/Bath/Table ($1,036,988.68), Sports/Leisure ($988,048.97), and Computers/Accessories ($911,954.32). No single category holds the bulk of revenue, as the combined share is below 50%.

**Sales Anomalies**
Between 2016-09-04 and 2018-10-17, 19 anomalous days were identified against a median daily payment of $23,559.52.
*   **High Anomalies:** 17 days exceeded the median. The highest was 2017-11-24 at $179,200.03. Other notable high days include 2017-11-25 ($71,897.01) and 2018-08-06 ($65,860.43).
*   **Low Anomalies:** 2 days fell below the median: 2017-

In [ ]:
# ask("Give me a full business report with recommendations.")

QUERY: Give me a full business report with recommendations.
USER REQUEST
 Give me a full business report with recommendations. 

ORCHESTRATOR DECISION
 SALES, CUSTOMER, PRODUCT, POLICY

TOOLS EXECUTED: ✅ sales, ✅ customer, ✅ product, ✅ policy

INSIGHT AGENT
 **Business Insight Report**

**Revenue & Customer Concentration**
Total revenue stands at $16,008,872.12 across 99,441 orders. Customer value is highly concentrated: Segment 3 represents only 0.9% of customers but generates 9.5% of revenue, with a value index of 10.56. Combined, high-value segments (1, 2, and 3) account for 12.1% of customers and 40.6% of revenue. In contrast, Segment 0 comprises 88.0% of customers but contributes only 59.5% of revenue, indicating a low value index of 0.68. Repeat customers are minimal, representing just 3.12% of the total customer base.

**Category Performance**
No single category dominates item sales. The top five categories collectively account for 39.74% of product item sales. Health & Beauty l

In [ ]:
# ask("Analyze customer segments and suggest retention policies.")

QUERY: Analyze customer segments and suggest retention policies.
USER REQUEST
 Analyze customer segments and suggest retention policies. 

ORCHESTRATOR DECISION
 CUSTOMER, POLICY

TOOLS EXECUTED: ✅ customer, ✅ policy

INSIGHT AGENT
 **Customer Segment Insights**

**Facts:**
*   **Segment 0** represents the majority of the customer base, comprising 88.0% of customers and 59.5% of revenue. Its value index is 0.68, indicating it generates less revenue relative to its size.
*   **Segment 1** accounts for 8.1% of customers but contributes 25.7% of revenue, with a value index of 3.17.
*   **Segment 2** comprises 3.1% of customers and 5.4% of revenue, with a value index of 1.74.
*   **Segment 3** is the smallest group at 0.9% of customers but drives 9.5% of revenue, showing the highest value index at 10.56.
*   High-value segments (those with a value index > 1) collectively represent 12.1% of customers and 40.6% of revenue.

**Interpretations:**
*   There is a significant disparity in custome

In [ ]:
# ask("What are our sales anomalies and what should we do about them per policy?")

QUERY: What are our sales anomalies and what should we do about them per policy?
USER REQUEST
 What are our sales anomalies and what should we do about them per policy? 

ORCHESTRATOR DECISION
 PRODUCT, POLICY

TOOLS EXECUTED: ✅ product, ✅ policy

INSIGHT AGENT
 **Business Insight Report**

**Category Performance**
The top five categories account for 39.74% of product item sales. Health & Beauty leads with $1,258,681.34, followed by Watches & Gifts ($1,205,005.68), Bed/Bath/Table ($1,036,988.68), Sports/Leisure ($988,048.97), and Computers/Accessories ($911,954.32). No single category holds the bulk of revenue, as the combined share is below 50%.

**Sales Anomalies**
Between 2016-09-04 and 2018-10-17, 19 anomalous days were identified against a median daily payment of $23,559.52.
*   **High Anomalies:** 17 days exceeded the median. The highest was 2017-11-24 at $179,200.03. Other notable high days include 2017-11-25 ($71,897.01) and 2018-08-06 ($65,860.43).
*   **Low Anomalies:** 2 day

other tests

In [ ]:
# ask("")

QUERY: (empty)
USER REQUEST
  

ORCHESTRATOR DECISION
 No request detected. Please provide your question or request so I can determine the appropriate tools.

TOOLS EXECUTED: 

POLICY ANSWER
 The question field is empty. Please provide a specific question regarding the policies to receive an answer.

RECOMMENDATIONS:
 The question field is empty. Please provide a specific question regarding the policies to receive an answer.


In [27]:
ask("asdkjhaskjdh")

QUERY: asdkjhaskjdh
USER REQUEST
 asdkjhaskjdh 

ORCHESTRATOR DECISION
 

TOOLS EXECUTED: 

POLICY ANSWER
 The policy text does not cover the request.

RECOMMENDATIONS:
 The policy text does not cover the request.


In [28]:
ask("What's the weather today?")

QUERY: What's the weather today?
USER REQUEST
 What's the weather today? 

ORCHESTRATOR DECISION
 

TOOLS EXECUTED: 

POLICY ANSWER
 The policy text does not contain any guidance on providing weather information. (No applicable policy)

RECOMMENDATIONS:
 The policy text does not contain any guidance on providing weather information. (No applicable policy)


In [30]:
ask("Tell me a joke.")

QUERY: Tell me a joke.
USER REQUEST
 Tell me a joke. 

ORCHESTRATOR DECISION
 

TOOLS EXECUTED: 

POLICY ANSWER
 The policy text does not cover providing jokes.

RECOMMENDATIONS:
 The policy text does not cover providing jokes.


In [ ]:
# ask("SALES")

QUERY: SALES
USER REQUEST
 SALES 

ORCHESTRATOR DECISION
 SALES

TOOLS EXECUTED: ✅ sales

INSIGHT AGENT
 **Business Insight Report**

**Key Facts**
*   **Scale:** Total revenue is $16,008,872.12 from 99,441 orders.
*   **Customer Base:** 96,096 total customers, with 2,997 repeat customers (3.12%).
*   **Order Value:** The average order value is $160.99.
*   **Category Performance:** The top five categories by product item sales are:
    1.  Health & Beauty: $1,258,681.34 (9.26% share)
    2.  Watches & Gifts: $1,205,005.68 (8.87% share)
    3.  Bed, Bath & Table: $1,036,988.68 (7.63% share)
    4.  Sports & Leisure: $988,048.97 (7.27% share)
    5.  Computers & Accessories: $911,954.32 (6.71% share)
*   **Concentration:** These top five categories combined account for $5,400,678.99 in item sales, representing 39.74% of total product item sales.

**Interpretations**
*   **Low Retention:** With only 3.12% of customers being repeat buyers, the business relies heavily on acquiring new cust

In [31]:
ask("Can you delete all customer data?")

QUERY: Can you delete all customer data?
USER REQUEST
 Can you delete all customer data? 

ORCHESTRATOR DECISION
 POLICY

TOOLS EXECUTED: ✅ policy

POLICY ANSWER
 Approval required – AI agents must not perform actions that need manager approval without proper authorization. (Policy 6.4)

RECOMMENDATIONS:
 Approval required – AI agents must not perform actions that need manager approval without proper authorization. (Policy 6.4)


In [32]:
ask("إيه أداء المبيعات بتاعنا؟")

QUERY: إيه أداء المبيعات بتاعنا؟
USER REQUEST
 إيه أداء المبيعات بتاعنا؟ 

ORCHESTRATOR DECISION
 SALES

TOOLS EXECUTED: ✅ sales

INSIGHT AGENT
 **Facts**  
- Total revenue: **$16,008,872.12**  
- Total orders: **99,441**  
- Total customers: **96,096**  
- Average order value (AOV): **$160.99**  
- Repeat customers: **2,997** (3.12 % of customers)  

**Top‑category item‑sales (share of product item sales)**  
- Health & Beauty: **$1,258,681.34** – **9.26 %**  
- Watches & Gifts: **$1,205,005.68** – **8.87 %**  
- Bed, Bath & Table: **$1,036,988.68** – **7.63 %**  
- Sports & Leisure: **$988,048.97** – **7.27 %**  
- Computers & Accessories: **$911,954.32** – **6.71 %**  

- Combined sales of the five categories: **$5,400,678.99**, representing **39.74 %** of item‑sales.  

**Interpretations**  
- The five leading categories together account for just under 40 % of item‑sales, indicating a fairly distributed product mix; no single category dominates the catalog.  
- Health & Beauty hold

In [33]:
ask("ممكن نعمل خصم 30%؟")

QUERY: ممكن نعمل خصم 30%؟
USER REQUEST
 ممكن نعمل خصم 30%؟ 

ORCHESTRATOR DECISION
 POLICY

TOOLS EXECUTED: ✅ policy

POLICY ANSWER
 Approval required – a 30 % discount exceeds the 20 % threshold and must have manager approval (Policy 3.2). Additionally, such an unusually large discount should be reviewed before application (Policy 3.3).

RECOMMENDATIONS:
 Approval required – a 30 % discount exceeds the 20 % threshold and must have manager approval (Policy 3.2). Additionally, such an unusually large discount should be reviewed before application (Policy 3.3).
